# Non-exponential memory: Prony stress test

This experiment gives classical Prony a kernel it did not generate: the completely monotone power-law kernel

$k(t)=(1+t/\tau)^{-\alpha}$, with $\alpha>0$ and $\tau>0$.

The goal is to test both kernel fidelity and the action of the MIN operator on a signal. Increasing the number of modes is not assumed to improve the result.


In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt
from min import MemoryOperator, fit_prony, power_law_kernel


In [ ]:
dt = 0.02
t = np.arange(401) * dt
alpha = 0.7
tau = 0.5
kernel_samples = power_law_kernel(t, alpha=alpha, tau=tau)
x = np.sin(2*np.pi*0.4*t) + 0.35*np.sin(2*np.pi*1.7*t)
kernel = lambda lag: power_law_kernel(lag, alpha=alpha, tau=tau)
reference = MemoryOperator(kernel).apply(t, x)


In [ ]:
def evaluate_fit(fit, t):
    t = np.asarray(t)
    return np.sum(fit.weights[:, None] * np.exp(-fit.gammas[:, None] * t[None, :]), axis=0)

def direct_from_kernel(t, x, kernel):
    return MemoryOperator(kernel).apply(t, x)

rows = []
for L in [2, 4, 8, 12, 16]:
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            fit = fit_prony(kernel_samples, dt, order=L)
            k_fit = evaluate_fit(fit, t)
            y_fit = direct_from_kernel(t, x, lambda lag, fit=fit: evaluate_fit(fit, lag))
        kernel_rel = np.linalg.norm(kernel_samples-k_fit) / np.linalg.norm(kernel_samples)
        operator_rel = np.linalg.norm(reference-y_fit) / np.linalg.norm(reference)
        imag = np.max(np.abs(np.imag(fit.gammas)))
        positive_real = np.sum((np.abs(np.imag(fit.gammas)) < 1e-8) & (np.real(fit.gammas) > 0))
        rows.append((L, kernel_rel, operator_rel, imag, positive_real, np.all(np.isfinite(k_fit))))
    except Exception as exc:
        rows.append((L, np.nan, np.nan, np.nan, np.nan, False))
        print(f'L={L}: {type(exc).__name__}: {exc}')

print(' L   kernel_rel      operator_rel      max|Im(gamma)|   positive-real-modes   finite')
for row in rows:
    print(f'{row[0]:2d}   {row[1]:12.4e}   {row[2]:14.4e}   {row[3]:15.4e}   {row[4]:19.0f}   {row[5]}')


## Interpretation

A failure of classical Prony here is a diagnostic result about identification and admissibility, not a failure of finite SOE memory itself. The next controlled comparison should keep the same power-law target and replace classical Prony with Hankel-SVD/ESPRIT, vector fitting, and finally a nonnegative SOE fit that explicitly respects complete monotonicity.

The experiment also records operator-action error because kernel approximation error and the error induced on a particular signal are different quantities.